In [1]:
import einops
import torch
import torch.nn as nn

In [40]:
x = torch.arange(start=0, end=16, step=1, dtype=torch.float32).view(2, 2, 4)
# x = torch.randn((4, 4))
C, H, W = x.shape
p = 2 # p^2 patch

N = (H // p) * (W // p)
# print(N)
print(x)


step1 = x.view(C, H // p, p, W // p , p)
print(step1)
print(step1.shape)
# C, H // p, p, W // p , p
# 0, 1,      2, 3,       4

step2 = step1.permute(1, 3, 0, 2, 4) # can now just view it!
print(step2)
print(step2.shape)

step3 = step2.reshape(-1, C * p * p)
print(step3)


tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.]],

        [[ 8.,  9., 10., 11.],
         [12., 13., 14., 15.]]])
tensor([[[[[ 0.,  1.],
           [ 2.,  3.]],

          [[ 4.,  5.],
           [ 6.,  7.]]]],



        [[[[ 8.,  9.],
           [10., 11.]],

          [[12., 13.],
           [14., 15.]]]]])
torch.Size([2, 1, 2, 2, 2])
tensor([[[[[ 0.,  1.],
           [ 4.,  5.]],

          [[ 8.,  9.],
           [12., 13.]]],


         [[[ 2.,  3.],
           [ 6.,  7.]],

          [[10., 11.],
           [14., 15.]]]]])
torch.Size([1, 2, 2, 2, 2])
tensor([[ 0.,  1.,  4.,  5.,  8.,  9., 12., 13.],
        [ 2.,  3.,  6.,  7., 10., 11., 14., 15.]])


In [29]:
def patchify2D(x, p, d):
    _, C, H, W = x.shape
    W = nn.Linear(C * p * p, d)
    x = einops.rearrange(x, "... c (h p1) (w p2) -> ...  (h w) (c p1 p2)", p1=p, p2=p)
    return W(x)

def patchify2Dpytorch(x, p, d):
    _, C, H, W = x.shape
    x = (x.view(C, H // p, p, W // p , p)
         .permute(1, 3, 0, 2, 4)
         .reshape(-1, C * p * p))
    
    W = nn.Linear(C * p * p, d)
    return W(x)

# conv is fastest
def patchify2DCONV(x, p, d):
    _, C, H, W = x.shape
    conv = nn.Conv2d(in_channels=C, out_channels=d, kernel_size=p, stride=p)
    # LINEAR 2x2x4 -> 2x8 (p=2) -> (2x8) @ (8 x d) -> 2xd 
    # CONV   CxHxW -> DxH/pxW/p -> DxN -> 2xNxD
    return einops.rearrange(conv(x), "... d h w -> ...  (h w) d")

In [ ]:
def DePatchify(x, p, d, c, h, w):
    # x # B, N, D -> B, H/p, W/p, D -> B, D, H/p, W/p
    # N = h // p * w // p
    x = einops.rearrange(x, "... (h1 w1) d ->... d h1 w1", h1 = h // p, w1 = w // p)
    deconv = nn.ConvTranspose2d(in_channels=d, out_channels=c, kernel_size=p, stride=p)
    return deconv(x)

In [ ]:
q = torch.arange(start=0, end=16, step=1, dtype=torch.float32).view(1, 2, 2, 4)
d = 4
DePatchify(patchify2DCONV(q, p=2, d=d), p=2, d=4, c=2, h=2, w=4)

torch.Size([1, 2, 2, 4])

In [ ]:
class ConvPatchify(nn.Module):
    def __init__(self, d_model: int, patch_size: int, n_channels: int = 4):
        """
        ConvPatchify
        d_model (int): model dimension
        patch_size (int): patch_size to split into
        n_channels (int): image channels

        input size: (B, C, X, Y)
        output size: (B, T, D)
        """
        super().__init__()

        self.conv = nn.Conv2d(
            n_channels, d_model, kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return einops.rearrange(self.conv(x), "d h w -> (h w) d")

Differences between `permute` and `view`.

So both can change the shapes/sizes/dimension of the tensor but they order them differently. 
* `view` works by taking the *contigous* tensor and taking each item as a 1D vector into the desired shape.
* `permute` works by taking into account the index ordering and reorders the tensor via indices non-contigously.
* `reshape` is like view but with `.contigous` built in the back-end if needed

Einops rearrange equivalency shown with PyTorch operations.

In [9]:
a = torch.arange(start=0, end=32, step=1).view(2, 4, 4)
print(a) # 2x4x4
print(a.view(4, 4, 2)) # contigous arrangement
print(a.permute(1, 2, 0)) # indexing kept CxHxW -> HxWxC also non contigous afterwards

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11],
         [12, 13, 14, 15]],

        [[16, 17, 18, 19],
         [20, 21, 22, 23],
         [24, 25, 26, 27],
         [28, 29, 30, 31]]])
tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5],
         [ 6,  7]],

        [[ 8,  9],
         [10, 11],
         [12, 13],
         [14, 15]],

        [[16, 17],
         [18, 19],
         [20, 21],
         [22, 23]],

        [[24, 25],
         [26, 27],
         [28, 29],
         [30, 31]]])
tensor([[[ 0, 16],
         [ 1, 17],
         [ 2, 18],
         [ 3, 19]],

        [[ 4, 20],
         [ 5, 21],
         [ 6, 22],
         [ 7, 23]],

        [[ 8, 24],
         [ 9, 25],
         [10, 26],
         [11, 27]],

        [[12, 28],
         [13, 29],
         [14, 30],
         [15, 31]]])


In [12]:
print(einops.rearrange(a, "c h w -> h w c") == a.permute(1, 2, 0))
m1 = a.permute(1, 0, 2).reshape(a.shape[1], a.shape[0]*a.shape[2])
b = einops.rearrange(a, "c h w -> h (c w)") # the same as permute + reshape

tensor([[[True, True],
         [True, True],
         [True, True],
         [True, True]],

        [[True, True],
         [True, True],
         [True, True],
         [True, True]],

        [[True, True],
         [True, True],
         [True, True],
         [True, True]],

        [[True, True],
         [True, True],
         [True, True],
         [True, True]]])
